
# Notebook 1 — SVM Hierárquico de Condições e Subcategorias SCP

## Objetivo

Treinar classificadores SVM independentes para:

- MI — Infarto do miocárdio
- STTC — Alterações ST/T
- CD — Distúrbios de condução
- HYP — Hipertrofias
- FORM — Alterações morfológicas

Além dos cinco classificadores gerais, cada código SCP
pertencente às categorias terá seu próprio classificador
binário quando houver dados suficientes.

## Características

- ECG bruto da PTB-XL, preferencialmente a 500 Hz
- 12 derivações preservadas individualmente
- Características estatísticas e espectrais
- Picos R e características RR estimadas
- Idade exata
- Sexo
- Quatro grupos etários

## Estratégia de classificação

Os códigos são multirrótulo. Um mesmo ECG pode conter
mais de um código SCP.

As probabilidades dos códigos são independentes e não
precisam somar 100%.

**Importante:** uma probabilidade estimada não equivale
a uma certeza diagnóstica.

## Estratégia de validação

- Folds 1–8: treinamento dos SVMs
- Fold 9: escolha de C e calibração
- Fold 10: avaliação independente

A seleção e a calibração no mesmo fold 9 podem produzir
estimativas otimistas nesse conjunto. O fold 10 permanece
intocado até a avaliação final.



## 1. Instalação das dependências

Execute no terminal do VS Code:

```powershell
python -m pip install numpy pandas scipy scikit-learn wfdb joblib matplotlib
```

O notebook utiliza `scipy.integrate.trapezoid`, evitando
a dependência de `numpy.trapezoid`.

Configure a pasta BASE na célula seguinte.


In [27]:

from pathlib import Path
import ast
import json
import warnings

import numpy as np
import pandas as pd
import scipy
from scipy import signal
from scipy.integrate import trapezoid

import wfdb
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, roc_auc_score,
    average_precision_score, brier_score_loss,
    confusion_matrix
)

SEED = 42

BASE = Path(r"C:\Users\Matheus\Documents\PTB-XL")

METADATA_CSV = BASE / "ptbxl_database.csv"
SCP_CSV = BASE / "scp_statements.csv"

DESTINO = Path(
    r"C:\Users\Matheus\Documents\Modelos\modelos_01"
)

CACHE = BASE / "resultados_svm_reestruturados" / (
    "cache_condicoes_subcategorias_v1.pkl"
)

DESTINO.mkdir(parents=True, exist_ok=True)
CACHE.parent.mkdir(parents=True, exist_ok=True)

# 0 = processar todos os registros.
MAX_REGISTROS = 0

# Controle de custo computacional do SVM RBF.
# 0 = sem limite por classe.
MAX_TREINO_POR_CLASSE = 2500

MIN_POS_TREINO = 30
MIN_POS_CALIBRACAO = 15
MIN_POS_TESTE = 5

C_CANDIDATOS = [0.3, 1.0, 3.0]

LEADS = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

GRUPOS = ["MI", "STTC", "CD", "HYP", "FORM"]

assert METADATA_CSV.exists(), METADATA_CSV
assert SCP_CSV.exists(), SCP_CSV

print("BASE:", BASE)
print("DESTINO:", DESTINO)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)


BASE: C:\Users\Matheus\Documents\PTB-XL
DESTINO: C:\Users\Matheus\Documents\Modelos\modelos_01
NumPy: 1.26.4
SciPy: 1.13.0



## 2. Leitura e auditoria da PTB-XL

Os códigos individuais são extraídos de `scp_codes`.

O `scp_statements.csv` estabelece a relação entre
códigos e categorias.

- MI, STTC, CD, HYP: coluna `diagnostic_class`
- FORM: coluna `form`

Um código pode participar de mais de uma dimensão
descritiva. Não se força exclusividade artificial.

Registros com NORM e diagnóstico anormal simultâneo
serão identificados e excluídos conforme a regra
utilizada anteriormente no projeto.


In [28]:

df = pd.read_csv(METADATA_CSV)
scp = pd.read_csv(SCP_CSV, index_col=0)

scp.index = scp.index.map(str)

def parse_codes(value):
    try:
        obj = ast.literal_eval(value) if isinstance(value, str) else value
        if isinstance(obj, dict):
            return set(str(k).strip() for k in obj)
    except (SyntaxError, ValueError, TypeError):
        pass
    return set()

df["code_set"] = df["scp_codes"].map(parse_codes)

MAPA = {g: set() for g in GRUPOS}
MAPA_CODIGO = {}

for code, row in scp.iterrows():
    code = str(code)
    membership = set()

    diagnostic_class = row.get("diagnostic_class")

    if pd.notna(diagnostic_class):
        group = str(diagnostic_class).strip()
        if group in GRUPOS:
            membership.add(group)

    form = row.get("form")
    if pd.notna(form) and float(form) == 1:
        membership.add("FORM")

    MAPA_CODIGO[code] = membership

    for group in membership:
        MAPA[group].add(code)

def categorias_registro(codes):
    result = set()
    for code in codes:
        result.update(MAPA_CODIGO.get(code, set()))
    return result

df["categories"] = df["code_set"].map(categorias_registro)

DIAGNOSTICOS = {"MI", "STTC", "CD", "HYP"}

df["conflito_norm"] = df.apply(
    lambda r:
        ("NORM" in r["code_set"])
        and bool(r["categories"] & DIAGNOSTICOS),
    axis=1
)

print("Total:", len(df))
print("Conflitos NORM/anormal:", int(df.conflito_norm.sum()))

for grupo in GRUPOS:
    print(grupo, "códigos:", len(MAPA[grupo]))

# Pacientes não devem cruzar os folds existentes.
cross = (
    df.groupby("patient_id", dropna=True)["strat_fold"]
      .nunique()
)

if (cross > 1).any():
    raise ValueError(
        "Existem pacientes em folds diferentes: "
        "verifique a divisão antes do treinamento."
    )

metadata = df.loc[
    ~df["conflito_norm"] & df["code_set"].map(bool)
].copy()

if MAX_REGISTROS:
    metadata = (
        metadata.groupby("strat_fold", group_keys=False)
        .apply(
            lambda x: x.sample(
                n=min(
                    len(x),
                    max(1, MAX_REGISTROS // 10)
                ),
                random_state=SEED
            )
        )
    )

display(
    pd.DataFrame([
        {
            "categoria": g,
            "quantidade_codigos": len(MAPA[g]),
            "codigos": ", ".join(sorted(MAPA[g]))
        }
        for g in GRUPOS
    ])
)


Total: 21799
Conflitos NORM/anormal: 445
MI códigos: 14
STTC códigos: 13
CD códigos: 11
HYP códigos: 5
FORM códigos: 19


,categoria,quantidade_codigos,codigos
0,MI,14,"ALMI, AMI, ASMI, ILMI, IMI, INJAL, INJAS, INJI..."
1,STTC,13,"ANEUR, DIG, EL, ISCAL, ISCAN, ISCAS, ISCIL, IS..."
2,CD,11,"1AVB, 2AVB, 3AVB, CLBBB, CRBBB, ILBBB, IRBBB, ..."
3,HYP,5,"LAO/LAE, LVH, RAO/RAE, RVH, SEHYP"
4,FORM,19,"ABQRS, DIG, HVOLT, INVT, LNGQT, LOWT, LPR, LVO..."



## 3. Extração de características dos sinais

São extraídas características por derivação, preservando
a identificação anatômica das 12 derivações.

Inclui:

- Desvio-padrão
- RMS
- Amplitude pico a pico
- MAD
- Variabilidade da primeira diferença
- Energia espectral em 1–5, 5–15 e 15–35 Hz
- Indicadores de qualidade e disponibilidade

A filtragem abaixo é apropriada para características
exploratórias, mas **não é utilizada para medir desvio
clínico de ST**.

A detecção de picos R é heurística e não equivale a
delineação clínica validada.


In [29]:

def clean_signal(x, fs):
    x = np.asarray(x, dtype=float)

    if len(x) < 3 * fs:
        return None

    good = np.isfinite(x)

    if good.mean() < 0.99:
        return None

    x = np.where(good, x, np.nanmedian(x[good]))

    hi = min(40.0, fs * 0.4)

    if hi <= 0.5:
        return None

    sos = signal.butter(
        3, [0.5, hi],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    return signal.sosfiltfilt(sos, x)


METRICAS_LEAD = [
    "std", "rms", "ptp", "mad",
    "deriv_mad", "energy_1_5",
    "energy_5_15", "energy_15_35"
]

def lead_features(raw, fs, lead):
    x = clean_signal(raw, fs)

    if x is None:
        out = {
            f"{lead}_{k}": np.nan
            for k in METRICAS_LEAD
        }
        out[f"{lead}_valid"] = 0
        return out

    out = {
        f"{lead}_valid": 1,
        f"{lead}_std": float(np.std(x)),
        f"{lead}_rms": float(np.sqrt(np.mean(x*x))),
        f"{lead}_ptp": float(np.ptp(x)),
        f"{lead}_mad": float(
            np.median(np.abs(x - np.median(x)))
        ),
        f"{lead}_deriv_mad": float(
            np.median(np.abs(np.diff(x)))
        )
    }

    freq, psd = signal.welch(
        x, fs=fs,
        nperseg=min(len(x), int(2*fs))
    )

    for a, b, name in [
        (1, 5, "energy_1_5"),
        (5, 15, "energy_5_15"),
        (15, 35, "energy_15_35")
    ]:
        mask = (freq >= a) & (freq < b)

        out[f"{lead}_{name}"] = (
            float(trapezoid(psd[mask], x=freq[mask]))
            if mask.sum() > 1
            else np.nan
        )

    return out


def rhythm_summary(x, fs):
    names = [
        "n_peaks", "hr_bpm", "rr_mean_ms",
        "rr_sd_ms", "rr_cv", "rmssd_ms",
        "rr_p10_ms", "rr_p90_ms"
    ]

    out = {k: np.nan for k in names}

    x = clean_signal(x, fs)

    if x is None:
        return out

    sos = signal.butter(
        2, [5, min(20, fs * 0.4)],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    y = signal.sosfiltfilt(sos, x)

    grad = np.gradient(y)

    width = max(3, int(0.08 * fs))

    energy = np.convolve(
        grad**2,
        np.ones(width) / width,
        mode="same"
    )

    peaks, _ = signal.find_peaks(
        energy,
        distance=max(1, int(0.28 * fs)),
        prominence=max(
            np.percentile(energy, 85) * 0.3,
            1e-12
        )
    )

    refined = []

    for p in peaks:
        left = max(0, p - int(0.075*fs))
        right = min(len(y), p + int(0.075*fs) + 1)

        if right > left:
            refined.append(
                left + int(np.argmax(np.abs(y[left:right])))
            )

    refined = np.unique(refined)

    if len(refined) > 1:
        keep = np.r_[
            True,
            np.diff(refined) > int(0.24 * fs)
        ]
        refined = refined[keep]

    rr = np.diff(refined) / fs

    rr = rr[(rr >= 0.28) & (rr <= 2.5)]

    if len(rr) < 4:
        return out

    out.update({
        "n_peaks": float(len(refined)),
        "hr_bpm": float(60 / np.mean(rr)),
        "rr_mean_ms": float(1000 * np.mean(rr)),
        "rr_sd_ms": float(1000 * np.std(rr)),
        "rr_cv": float(np.std(rr) / np.mean(rr)),
        "rmssd_ms": float(
            1000 * np.sqrt(np.mean(np.diff(rr)**2))
        ),
        "rr_p10_ms": float(1000 * np.quantile(rr, .1)),
        "rr_p90_ms": float(1000 * np.quantile(rr, .9))
    })

    return out


def extract_record(record):
    x = np.asarray(record.p_signal, dtype=float)
    fs = float(record.fs)

    names = list(record.sig_name)

    if x.ndim != 2 or x.shape[0] < int(5 * fs):
        return None

    out = {}

    for lead in LEADS:
        if lead in names:
            raw = x[:, names.index(lead)]
        else:
            raw = np.array([])

        out.update(lead_features(raw, fs, lead))

    reference = (
        "II" if "II" in names
        else ("I" if "I" in names else None)
    )

    if reference:
        out.update(
            rhythm_summary(x[:, names.index(reference)], fs)
        )
    else:
        out.update(
            rhythm_summary(np.array([]), fs)
        )

    return out

print("Extrator definido.")


Extrator definido.



## 4. Extração e cache

O cache guarda características derivadas do sinal,
sem modificar os arquivos WFDB originais.

O contexto de idade e sexo é aplicado posteriormente,
para permitir simulações com os mesmos sinais.

A primeira execução processará os sinais disponíveis.
As execuções seguintes poderão reutilizar o cache.


In [30]:

def build_cache(meta, path):
    if path.exists():
        cached = pd.read_pickle(path)

        if (
            isinstance(cached, pd.DataFrame)
            and "ecg_id" in cached.columns
            and not cached.empty
        ):
            print("Cache recuperado:", len(cached))
            return cached

    rows = []
    failures = []

    for i, (_, row) in enumerate(meta.iterrows(), 1):
        record = None

        for field in ["filename_hr", "filename_lr"]:
            filename = row.get(field)

            if not isinstance(filename, str):
                continue

            prefix = BASE / filename

            if not prefix.with_suffix(".hea").exists():
                continue

            try:
                record = wfdb.rdrecord(str(prefix))
                break
            except Exception:
                continue

        if record is None:
            failures.append(
                (row.ecg_id, "WFDB indisponível")
            )
            continue

        try:
            feat = extract_record(record)
        except Exception as exc:
            failures.append(
                (row.ecg_id, str(exc))
            )
            continue

        if feat is None:
            failures.append(
                (row.ecg_id, "sinal inválido")
            )
            continue

        feat["ecg_id"] = int(row.ecg_id)
        rows.append(feat)

        if i % 500 == 0:
            print(
                f"{i}/{len(meta)}; válidos={len(rows)}",
                flush=True
            )

    pd.DataFrame(
        failures,
        columns=["ecg_id", "motivo"]
    ).to_csv(
        path.with_suffix(".falhas.csv"),
        index=False
    )

    result = pd.DataFrame(rows)

    if result.empty:
        raise RuntimeError(
            "Nenhum sinal foi processado. "
            "Verifique records500/ e records100/."
        )

    result.to_pickle(path)

    print("Registros aproveitados:", len(result))

    return result


assert (
    (BASE / "records500").exists()
    or (BASE / "records100").exists()
), "Não foram encontradas pastas de sinais WFDB."

features_table = build_cache(metadata, CACHE)


Cache recuperado: 21354



## 5. Idade, sexo e faixas etárias

### Grupos da interface

- Pediátrico: 0–16
- Adulto jovem: 17–39
- Adulto: 40–59
- Idoso: 60+

A idade exata também permanece como característica.

Na PTB-XL:

- `sex = 0`: masculino
- `sex = 1`: feminino

As regras exclusivas para adultos utilizam idade
mínima de 18 anos, mesmo quando a interface agrupa
a idade de 17 anos em adulto jovem.

O modelo poderá aprender interações demográficas,
mas não se força uma mudança artificial nas probabilidades.

### Regras clínicas

O motor clínico original descreve limites para
FC, PR, QRS, QTc e ST.

Este notebook só extrai diretamente características
exploratórias de sinal e RR. Não possui delineação
clínica validada de PR, QRS, QTc e ponto J.

Consequentemente, não inventará essas medidas nem
transformará ausência de medição em normalidade.

As regras clínicas completas devem permanecer
separadas do SVM até existirem medidas válidas.


In [31]:

def age_group(age):
    if pd.isna(age) or age < 0 or age > 120:
        return "desconhecido"
    if age <= 16:
        return "pediatrico_0_16"
    if age <= 39:
        return "adulto_jovem_17_39"
    if age <= 59:
        return "adulto_40_59"
    return "idoso_60_mais"


AGE_GROUPS = [
    "pediatrico_0_16",
    "adulto_jovem_17_39",
    "adulto_40_59",
    "idoso_60_mais",
    "desconhecido"
]


def contextualize(frame):
    d = frame.copy()

    age = pd.to_numeric(
        d["age"], errors="coerce"
    )

    sex = pd.to_numeric(
        d["sex_ptbxl"], errors="coerce"
    )

    groups = age.map(age_group)

    d["age_missing"] = age.isna().astype(int)

    d["sex_masculino"] = (sex == 0).astype(int)
    d["sex_feminino"] = (sex == 1).astype(int)

    d["sex_desconhecido"] = (
        ~sex.isin([0, 1])
    ).astype(int)

    d["idade_maior_igual_18"] = (
        age >= 18
    ).astype(int)

    for group in AGE_GROUPS:
        d["faixa_" + group] = (
            groups == group
        ).astype(int)

    return d


merged = features_table.merge(
    metadata[
        [
            "ecg_id", "patient_id",
            "age", "sex", "strat_fold",
            "code_set", "categories"
        ]
    ],
    on="ecg_id",
    how="inner",
    validate="one_to_one"
)

merged = merged.rename(
    columns={"sex": "sex_ptbxl"}
)

merged = contextualize(merged)

IGNORAR = {
    "ecg_id", "patient_id", "strat_fold",
    "code_set", "categories"
}

FEATURE_COLS = [
    c for c in merged.columns
    if c not in IGNORAR
    and pd.api.types.is_numeric_dtype(merged[c])
]

print("ECGs:", len(merged))
print("Características:", len(FEATURE_COLS))
print("Folds:", merged.strat_fold.value_counts().sort_index())


ECGs: 21354
Características: 128
Folds: strat_fold
1     2133
2     2139
3     2142
4     2132
5     2128
6     2133
7     2128
8     2130
9     2142
10    2147
Name: count, dtype: int64



## 6. Alvos hierárquicos e multirrótulo

Esta etapa corrige a limitação do notebook anterior.

Serão construídos:

1. Cinco alvos gerais.
2. Alvos individuais para os códigos SCP de cada grupo.

O mapeamento real é obtido do `scp_statements.csv`.

Exemplo de nomenclatura:

- `MI`
- `MI__IMI`
- `MI__ASMI`
- `CD__CRBBB`

Os nomes representam tarefas distintas.

O mesmo código SCP pode pertencer a mais de uma
dimensão documentada. Essa informação é preservada.

Um modelo somente será treinado se houver exemplos
positivos e negativos suficientes nos folds exigidos.


In [32]:

LABELS = {}
LABEL_INFO = {}

# Cinco classificadores gerais
for group in GRUPOS:
    LABELS[group] = merged["categories"].map(
        lambda groups: int(group in groups)
    ).to_numpy(dtype=int)

    LABEL_INFO[group] = {
        "tipo": "categoria",
        "categoria": group,
        "codigo_scp": None
    }

# Subcategorias individuais
for group in GRUPOS:
    for code in sorted(MAPA[group]):

        name = f"{group}__{code}"

        LABELS[name] = merged["code_set"].map(
            lambda codes, code=code:
                int(code in codes)
        ).to_numpy(dtype=int)

        LABEL_INFO[name] = {
            "tipo": "subcategoria",
            "categoria": group,
            "codigo_scp": code
        }

audit = []

for label, y in LABELS.items():
    item = LABEL_INFO[label]

    audit.append({
        "modelo": label,
        "categoria": item["categoria"],
        "tipo": item["tipo"],
        "codigo_scp": item["codigo_scp"],
        "positivos": int(y.sum()),
        "negativos": int((1-y).sum())
    })

audit = pd.DataFrame(audit)

audit.to_csv(
    DESTINO / "auditoria_alvos.csv",
    index=False
)

display(audit)
print("Total de tarefas candidatas:", len(LABELS))


,modelo,categoria,tipo,codigo_scp,positivos,negativos
0,MI,MI,categoria,None,5468,15886
1,STTC,STTC,categoria,None,5202,16152
2,CD,CD,categoria,None,4483,16871
3,HYP,HYP,categoria,None,2644,18710
4,FORM,FORM,categoria,None,8888,12466
...,...,...,...,...,...,...
62,FORM__QWAVE,FORM,subcategoria,QWAVE,540,20814
63,FORM__STD_,FORM,subcategoria,STD_,1004,20350
64,FORM__STE_,FORM,subcategoria,STE_,28,21326
65,FORM__TAB_,FORM,subcategoria,TAB_,29,21325


Total de tarefas candidatas: 67



## 7. Treinamento SVM e calibração

### SVM

Cada alvo terá um classificador binário SVC com:

- Kernel RBF
- `class_weight="balanced"`
- Imputação mediana
- Padronização das variáveis
- Valores candidatos de C: 0,3; 1; 3

A pesquisa de C utilizará o fold 9.

### Probabilidades

O `decision_function()` do SVC não representa uma
probabilidade.

Para obter percentuais será utilizada calibração
sigmoidal de Platt:

1. Treinar SVM nos folds 1–8.
2. Obter escores no fold 9.
3. Ajustar uma regressão logística de calibração.
4. Aplicar SVM e calibrador no fold 10.

A calibração e a escolha de C compartilham o fold 9,
portanto o desempenho nesse fold é exploratório.

O desempenho definitivo será reportado no fold 10.

O calibrador aprende probabilidades relativas aos
rótulos anotados, não certezas clínicas.


In [33]:

def pipeline_svm(C):
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=True
            )
        ),
        ("scaler", StandardScaler()),
        (
            "svm",
            SVC(
                kernel="rbf",
                C=C,
                gamma="scale",
                class_weight="balanced",
                probability=False,
                random_state=SEED
            )
        )
    ])


def sample_train(y, indices, cap, seed):
    rng = np.random.default_rng(seed)
    selected = []

    for value in [0, 1]:
        ids = indices[y[indices] == value]

        if cap and len(ids) > cap:
            ids = rng.choice(
                ids, size=cap, replace=False
            )

        selected.extend(ids.tolist())

    return np.asarray(
        sorted(selected), dtype=int
    )


def metricas(y, probability, threshold=0.5):
    pred = (probability >= threshold).astype(int)

    cm = confusion_matrix(
        y, pred, labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    result = {
        "n": int(len(y)),
        "positivos": int(y.sum()),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y, pred)
        ),
        "precision": float(
            precision_score(y, pred, zero_division=0)
        ),
        "recall": float(
            recall_score(y, pred, zero_division=0)
        ),
        "f1": float(
            f1_score(y, pred, zero_division=0)
        ),
        "specificity": (
            float(tn / (tn + fp))
            if tn + fp else np.nan
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }

    if len(np.unique(y)) == 2:
        result["roc_auc"] = float(
            roc_auc_score(y, probability)
        )
        result["average_precision"] = float(
            average_precision_score(y, probability)
        )
        result["brier_score"] = float(
            brier_score_loss(y, probability)
        )
    else:
        result.update({
            "roc_auc": np.nan,
            "average_precision": np.nan,
            "brier_score": np.nan
        })

    return result


def platt_fit(scores, targets):
    calibrator = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=SEED
    )

    calibrator.fit(
        np.asarray(scores).reshape(-1, 1),
        targets
    )

    return calibrator


def probability_from_scores(calibrator, scores):
    return calibrator.predict_proba(
        np.asarray(scores).reshape(-1, 1)
    )[:, 1]


print("Treinamento e calibração definidos.")


Treinamento e calibração definidos.



## 8. Treinar os classificadores gerais e específicos

Para cada alvo:

1. Confirmar quantidade de amostras.
2. Selecionar hiperparâmetro usando fold 9.
3. Ajustar o calibrador.
4. Avaliar no fold 10.
5. Salvar o pipeline e o calibrador.
6. Salvar a lista de características.
7. Registrar a categoria e o código SCP.

Modelos ignorados serão documentados separadamente.

**Atenção ao custo:** SVM RBF com muitos alvos pode
exigir bastante memória e processamento. A configuração
de amostragem do treinamento pode ser ajustada.


In [34]:

# ============================================================
# AUDITORIA COMPLETA DAS SUBCATEGORIAS SCP
# NÃO TREINA NOVAMENTE OS MODELOS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

DESTINO = Path(
    r"C:\Users\Matheus\Documents\Modelos\modelos_01"
)
DESTINO.mkdir(parents=True, exist_ok=True)

folds = merged["strat_fold"].to_numpy(dtype=int)

train_idx = np.flatnonzero(
    (folds >= 1) & (folds <= 8)
)
valid_idx = np.flatnonzero(folds == 9)
test_idx = np.flatnonzero(folds == 10)

registros = []

for label, y in LABELS.items():
    y = np.asarray(y, dtype=int)
    info = LABEL_INFO[label]

    treino = int(y[train_idx].sum())
    validacao = int(y[valid_idx].sum())
    teste = int(y[test_idx].sum())

    neg_treino = int(len(train_idx) - treino)
    neg_validacao = int(len(valid_idx) - validacao)
    neg_teste = int(len(test_idx) - teste)

    motivos = []

    if treino < MIN_POS_TREINO:
        motivos.append("Poucos positivos no treino")

    if neg_treino < 2:
        motivos.append("Poucos negativos no treino")

    if validacao < max(9, MIN_POS_CALIBRACAO):
        motivos.append("Poucos positivos na validação")

    if neg_validacao < 9:
        motivos.append("Poucos negativos na validação")

    if teste < MIN_POS_TESTE:
        motivos.append("Poucos positivos no teste")

    if neg_teste < 1:
        motivos.append("Sem negativos no teste")

    registros.append({
        "modelo": label,
        "grupo": info["categoria"],
        "tipo": info["tipo"],
        "codigo_scp": info["codigo_scp"],
        "total_positivos": int(y.sum()),
        "positivos_treino": treino,
        "positivos_validacao": validacao,
        "positivos_teste": teste,
        "negativos_treino": neg_treino,
        "negativos_validacao": neg_validacao,
        "negativos_teste": neg_teste,
        "elegivel_pelos_filtros_atuais": (
            len(motivos) == 0
        ),
        "motivo": (
            "; ".join(motivos)
            if motivos
            else "Atende aos critérios"
        )
    })

auditoria = pd.DataFrame(registros)

auditoria.to_csv(
    DESTINO / "auditoria_completa_subcategorias.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total de classificadores candidatos:", len(auditoria))

print(
    "Elegíveis pelos critérios atuais:",
    int(auditoria["elegivel_pelos_filtros_atuais"].sum())
)

print(
    "Ignorados pelos critérios atuais:",
    int((~auditoria["elegivel_pelos_filtros_atuais"]).sum())
)

print("\nAUDITORIA DE HYP")
display(
    auditoria[
        auditoria["grupo"] == "HYP"
    ]
)

print("\nRESUMO DE TODAS AS CATEGORIAS")
display(
    auditoria.groupby("grupo").agg(
        identificados=("modelo", "count"),
        elegiveis=(
            "elegivel_pelos_filtros_atuais",
            "sum"
        )
    )
)

print("\nArquivo exportado:")
print(
    DESTINO / "auditoria_completa_subcategorias.csv"
)


Total de classificadores candidatos: 67
Elegíveis pelos critérios atuais: 44
Ignorados pelos critérios atuais: 23

AUDITORIA DE HYP


,modelo,grupo,tipo,codigo_scp,total_positivos,positivos_treino,positivos_validacao,positivos_teste,negativos_treino,negativos_validacao,negativos_teste,elegivel_pelos_filtros_atuais,motivo
3,HYP,HYP,categoria,None,2644,2117,267,260,14948,1875,1887,True,Atende aos critérios
43,HYP__LAO/LAE,HYP,subcategoria,LAO/LAE,423,340,43,40,16725,2099,2107,True,Atende aos critérios
44,HYP__LVH,HYP,subcategoria,LVH,2131,1707,210,214,15358,1932,1933,True,Atende aos critérios
45,HYP__RAO/RAE,HYP,subcategoria,RAO/RAE,98,79,9,10,16986,2133,2137,False,Poucos positivos na validação
46,HYP__RVH,HYP,subcategoria,RVH,126,102,12,12,16963,2130,2135,False,Poucos positivos na validação
47,HYP__SEHYP,HYP,subcategoria,SEHYP,29,24,3,2,17041,2139,2145,False,Poucos positivos no treino; Poucos positivos n...



RESUMO DE TODAS AS CATEGORIAS


,identificados,elegiveis
grupo,,
CD,12,8
FORM,20,15
HYP,6,3
MI,15,9
STTC,14,9



Arquivo exportado:
C:\Users\Matheus\Documents\Modelos\modelos_01\auditoria_completa_subcategorias.csv


In [ ]:

# ============================================================
# CÉLULA 16 — TREINAMENTO SVM CORRIGIDO
# CATEGORIAS PRINCIPAIS + SUBCATEGORIAS SCP
# ============================================================
#
# Correções:
# 1. Nomes de arquivos compatíveis com Windows
# 2. Preservação dos códigos SCP originais
# 3. Verificação de nomes duplicados
# 4. Limiar individual otimizado por F1
# 5. Probabilidades calibradas
# 6. Relatórios salvos progressivamente
# 7. Idade, sexo e faixas etárias preservados
# 8. Fold 10 reservado exclusivamente para teste
#
# ============================================================

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd
import joblib
import json
import re
import hashlib

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

DESTINO = Path(
    r"C:\Users\Matheus\Documents\Modelos\modelos_01"
)

DESTINO.mkdir(
    parents=True,
    exist_ok=True
)

C_CANDIDATOS = [0.3, 1.0, 3.0]

# Conferir dependências definidas nas células anteriores
NECESSARIOS = [
    "merged",
    "FEATURE_COLS",
    "LABELS",
    "LABEL_INFO",
    "AGE_GROUPS",
    "SEED",
    "MIN_POS_TREINO",
    "MIN_POS_CALIBRACAO",
    "MIN_POS_TESTE",
    "MAX_TREINO_POR_CLASSE",
    "sample_train",
    "pipeline_svm",
    "platt_fit",
    "probability_from_scores",
    "metricas",
    "average_precision_score"
]

faltando = [
    nome for nome in NECESSARIOS
    if nome not in globals()
]

if faltando:
    raise RuntimeError(
        "Execute as células anteriores. "
        f"Variáveis ou funções ausentes: {faltando}"
    )

# ============================================================
# 2. CORRIGIR NOMES DE ARQUIVOS
# ============================================================

def nome_arquivo_seguro(label):
    """
    Remove caracteres inválidos em nomes Windows.

    Exemplo:
    HYP__LAO/LAE -> HYP__LAO_LAE

    Não modifica o rótulo SCP original.
    """

    nome = str(label).strip()

    nome = re.sub(
        r'[<>:"/\\|?*\x00-\x1f]',
        "_",
        nome
    )

    nome = re.sub(
        r"_+",
        "_",
        nome
    )

    nome = nome.rstrip(" .")

    if not nome:
        nome = "modelo_sem_nome"

    return nome


# Construir nomes antes de iniciar o treinamento.
# Usar hash apenas em caso de colisão.
NOMES_ARQUIVOS = {}

nomes_usados = set()

for label in LABELS:

    nome = nome_arquivo_seguro(label)

    chave = nome.casefold()

    if chave in nomes_usados:

        sufixo = hashlib.sha256(
            str(label).encode("utf-8")
        ).hexdigest()[:8]

        nome = f"{nome}_{sufixo}"

        chave = nome.casefold()

    if chave in nomes_usados:
        raise RuntimeError(
            f"Colisão no nome do modelo: {label}"
        )

    nomes_usados.add(chave)

    NOMES_ARQUIVOS[label] = f"{nome}.joblib"

# Conferência dos nomes problemáticos
print("=" * 65)
print("AUDITORIA DOS NOMES DE ARQUIVOS")
print("=" * 65)

for label, filename in NOMES_ARQUIVOS.items():

    if filename != f"{label}.joblib":
        print(f"{label} -> {filename}")

# ============================================================
# 3. PREPARAÇÃO DOS DADOS
# ============================================================

X = merged[FEATURE_COLS].to_numpy(
    dtype=float
)

folds = merged[
    "strat_fold"
].to_numpy(dtype=int)

TRAIN = np.flatnonzero(
    (folds >= 1) & (folds <= 8)
)

VALID = np.flatnonzero(
    folds == 9
)

TEST = np.flatnonzero(
    folds == 10
)

assert all(
    len(indices) > 0
    for indices in [TRAIN, VALID, TEST]
), "Treino, validação ou teste está vazio."


# ============================================================
# 4. VERIFICAR DIVISÃO POR PACIENTE
# ============================================================

if "patient_id" in merged.columns:

    def pacientes(indices):
        return set(
            merged.iloc[indices][
                "patient_id"
            ].dropna()
        )

    pt_train = pacientes(TRAIN)
    pt_valid = pacientes(VALID)
    pt_test = pacientes(TEST)

    assert not (pt_train & pt_valid), (
        "Paciente presente no treino e validação."
    )

    assert not (pt_train & pt_test), (
        "Paciente presente no treino e teste."
    )

    assert not (pt_valid & pt_test), (
        "Paciente presente na validação e teste."
    )


print("\n" + "=" * 65)
print("CONFIGURAÇÃO DO TREINAMENTO")
print("=" * 65)

print("Treino:", len(TRAIN))
print("Validação:", len(VALID))
print("Teste:", len(TEST))

print(
    "Número de características:",
    len(FEATURE_COLS)
)

print(
    "Quantidade de modelos candidatos:",
    len(LABELS)
)

print("Destino:", DESTINO.resolve())


# ============================================================
# 5. OTIMIZAR LIMIAR DE CLASSIFICAÇÃO
# ============================================================

def otimizar_limiar_f1(y_true, probabilities):
    """
    Escolhe o limiar individual que maximiza F1.

    Utiliza apenas a parcela reservada do fold 9.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )

    if (
        len(y_true) == 0
        or len(np.unique(y_true)) < 2
        or not np.isfinite(probabilities).all()
    ):
        return {
            "threshold": 0.5,
            "f1_validacao": np.nan,
            "precision_validacao": np.nan,
            "recall_validacao": np.nan
        }

    precision, recall, thresholds = (
        precision_recall_curve(
            y_true,
            probabilities
        )
    )

    if len(thresholds) == 0:
        return {
            "threshold": 0.5,
            "f1_validacao": np.nan,
            "precision_validacao": np.nan,
            "recall_validacao": np.nan
        }

    p = precision[:-1]
    r = recall[:-1]

    f1 = (
        2 * p * r
        / np.maximum(p + r, 1e-12)
    )

    melhor_f1 = np.max(f1)

    candidatos = np.flatnonzero(
        np.isclose(f1, melhor_f1)
    )

    # Em caso de empate, favorecer precisão
    melhor = candidatos[
        np.argmax(p[candidatos])
    ]

    return {
        "threshold": float(
            thresholds[melhor]
        ),
        "f1_validacao": float(
            f1[melhor]
        ),
        "precision_validacao": float(
            p[melhor]
        ),
        "recall_validacao": float(
            r[melhor]
        )
    }


# ============================================================
# 6. DIVISÃO INTERNA DO FOLD 9
# ============================================================

def separar_validacao(indices, y, seed):
    """
    Fold 9 dividido em:

    40%: seleção do C
    30%: calibração
    30%: otimização do limiar
    """

    indices = np.asarray(
        indices,
        dtype=int
    )

    select_idx, restante = train_test_split(
        indices,
        test_size=0.60,
        stratify=y[indices],
        random_state=seed
    )

    calib_idx, threshold_idx = (
        train_test_split(
            restante,
            test_size=0.50,
            stratify=y[restante],
            random_state=seed + 1
        )
    )

    return (
        select_idx,
        calib_idx,
        threshold_idx
    )


# ============================================================
# 7. RELATÓRIOS E CHECKPOINTS
# ============================================================

REPORT = []
IGNORADOS = []


def salvar_relatorios():
    """
    Salva os relatórios ao longo do treinamento.

    Caso ocorra uma interrupção, os resultados
    já concluídos permanecem documentados.
    """

    pd.DataFrame(REPORT).to_csv(
        DESTINO / "metricas_todos_modelos.csv",
        index=False
    )

    pd.DataFrame(IGNORADOS).to_csv(
        DESTINO / "modelos_nao_treinados.csv",
        index=False
    )

    configuracao = {
        "total_candidatos": len(LABELS),
        "total_treinados": len(REPORT),
        "total_ignorados": len(IGNORADOS),
        "feature_cols": list(FEATURE_COLS),
        "age_groups": list(AGE_GROUPS),
        "sex_encoding": {
            "masculino": 0,
            "feminino": 1
        },
        "folds": {
            "treino": list(range(1, 9)),
            "validacao": 9,
            "teste": 10
        },
        "arquivos": {
            item["modelo"]: item["arquivo"]
            for item in REPORT
        }
    }

    (
        DESTINO / "configuracao_treinamenf
        json.dumps(
            configuracao,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


# ============================================================
# 8. TREINAMENTO DOS MODELOS
# ============================================================

print("\nIniciando treinamento...\n")

for k, (label, y) in enumerate(
    LABELS.items(),
    1
):

    y = np.asarray(y, dtype=int)

    n_train = int(y[TRAIN].sum())
    n_valid = int(y[VALID].sum())
    n_test = int(y[TEST].sum())

    n_train_neg = int(
        len(TRAIN) - n_train
    )

    n_valid_neg = int(
        len(VALID) - n_valid
    )

    n_test_neg = int(
        len(TEST) - n_test
    )

    print("\n" + "-" * 65)
    print(f"[{k}/{len(LABELS)}] {label}")
    print("-" * 65)

    print(
        f"Positivos: treino={n_train}, "
        f"validação={n_valid}, "
        f"teste={n_test}"
    )

    # --------------------------------------------------------
    # 8.1. Verificar elegibilidade
    # --------------------------------------------------------

    if (
        n_train < MIN_POS_TREINO
        or n_train_neg < 2
        or n_valid < max(
            9,
            MIN_POS_CALIBRACAO
        )
        or n_valid_neg < 9
        or n_test < MIN_POS_TESTE
        or n_test_neg < 1
    ):

        IGNORADOS.append({
            "modelo": label,
            "motivo": "Amostras insuficientes",
            "train_pos": n_train,
            "valid_pos": n_valid,
            "test_pos": n_test
        })

        salvar_relatorios()

        print("Modelo ignorado.")
        continue

    # --------------------------------------------------------
    # 8.2. Amostragem somente no treinamento
    # --------------------------------------------------------

    chosen_train = sample_train(
        y,
        TRAIN,
        MAX_TREINO_POR_CLASSE,
        SEED
    )

    # --------------------------------------------------------
    # 8.3. Separar validação
    # --------------------------------------------------------

    try:

        (
            IDX_SELECT,
            IDX_CALIB,
            IDX_THRESHOLD
        ) = separar_validacao(
            VALID,
            y,
            SEED
        )

    except ValueError as exc:

        IGNORADOS.append({
            "modelo": label,
            "motivo": str(exc),
            "train_pos": n_train,
            "valid_pos": n_valid,
            "test_pos": n_test
        })

        salvar_relatorios()

        print(
            "Modelo ignorado:",
            exc
        )
        continue

    partes = [
        IDX_SELECT,
        IDX_CALIB,
        IDX_THRESHOLD
    ]

    if any(
        len(np.unique(y[idx])) < 2
        for idx in partes
    ):

        IGNORADOS.append({
            "modelo": label,
            "motivo": (
                "Falta uma classe na validação"
            ),
            "train_pos": n_train,
            "valid_pos": n_valid,
            "test_pos": n_test
        })

        salvar_relatorios()

        continue

    # --------------------------------------------------------
    # 8.4. Seleção do melhor C
    # --------------------------------------------------------

    best = None

    for C in C_CANDIDATOS:

        candidate = pipeline_svm(C)

        candidate.fit(
            X[chosen_train],
            y[chosen_train]
        )

        scores = (
            candidate.decision_function(
                X[IDX_SELECT]
            )
        )

        ap = average_precision_score(
            y[IDX_SELECT],
            scores
        )

        if (
            best is None
            or ap > best["ap"]
        ):

            best = {
                "model": candidate,
                "C": C,
                "ap": float(ap)
            }

    # --------------------------------------------------------
    # 8.5. Calibração das probabilidades
    # --------------------------------------------------------

    scores_calib = (
        best["model"].decision_function(
            X[IDX_CALIB]
        )
    )

    calibrator = platt_fit(
        scores_calib,
        y[IDX_CALIB]
    )

    # --------------------------------------------------------
    # 8.6. Limiar individual
    # --------------------------------------------------------

    scores_threshold = (
        best["model"].decision_function(
            X[IDX_THRESHOLD]
        )
    )

    probabilities_threshold = (
        probability_from_scores(
            calibrator,
            scores_threshold
        )
    )

    threshold_info = otimizar_limiar_f1(
        y[IDX_THRESHOLD],
        probabilities_threshold
    )

    threshold = threshold_info[
        "threshold"
    ]

    # --------------------------------------------------------
    # 8.7. Teste independente
    # --------------------------------------------------------

    test_scores = (
        best["model"].decision_function(
            X[TEST]
        )
    )

    test_prob = (
        probability_from_scores(
            calibrator,
            test_scores
        )
    )

    metrics = metricas(
        y[TEST],
        test_prob,
        threshold=threshold
    )

    # --------------------------------------------------------
    # 8.8. Metadados da categoria
    # --------------------------------------------------------

    info = LABEL_INFO[label]

    # --------------------------------------------------------
    # 8.9. Criar artefato do modelo
    # --------------------------------------------------------

    artifact = {
        "model": best["model"],
        "calibrator": calibrator,

        "features": list(FEATURE_COLS),

        # Preserva o nome SCP original
        "label": str(label),

        "task": "condicoes_e_subcategorias",

        "categoria": info["categoria"],
        "tipo": info["tipo"],
        "codigo_scp": info["codigo_scp"],

        "threshold": float(threshold),

        "C": best["C"],

        "calibration": "platt_logistic_fold9",
        "probability_output": True,

        "train_folds": list(range(1, 9)),
        "calibration_fold": 9,
        "test_fold": 10,

        "validation_partition": {
            "selection": "40% do fold 9",
            "calibration": "30% do fold 9",
            "threshold": "30% do fold 9"
        },

        "sex_encoding": {
            "masculino": 0,
            "feminino": 1
        },

        "age_groups": list(AGE_GROUPS),

        "threshold_selection": (
            "maximize_f1_validation"
        ),

        "note": (
            "Probabilidade calibrada de rótulo SCP. "
            "Não representa diagnóstico clínico."
        )
    }

    # --------------------------------------------------------
    # 8.10. NOME CORRIGIDO DO ARQUIVO
    # --------------------------------------------------------

    # ANTES:
    # filename = f"{label}.joblib"
    #
    # ERRO:
    # HYP__LAO/LAE.joblib
    #
    # AGORA:
    # HYP__LAO_LAE.joblib

    filename = NOMES_ARQUIVOS[label]

    caminho_modelo = DESTINO / filename

    # Garantir que a pasta exista
    caminho_modelo.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Preservar o nome do arquivo no artefato
    artifact["arquivo_modelo"] = filename

    # --------------------------------------------------------
    # 8.11. SALVAR MODELO
    # --------------------------------------------------------

    joblib.dump(
        artifact,
        caminho_modelo,
        compress=3
    )

    if not caminho_modelo.is_file():
        raise RuntimeError(
            f"Falha ao salvar: {caminho_modelo}"
        )

    # --------------------------------------------------------
    # 8.12. Registrar resultados
    # --------------------------------------------------------

    REPORT.append({
        "modelo": label,

        **info,

        "arquivo": filename,

        "C": best["C"],

        "threshold": float(threshold),

        "train_pos": n_train,
        "valid_pos": n_valid,
        "test_pos": n_test,

        "fold9_ap_selecao": best["ap"],

        "fold9_f1_threshold": (
            threshold_info["f1_validacao"]
        ),

        **metrics
    })

    # Salvar os relatórios imediatamente
    salvar_relatorios()

    print(
        f"F1 TESTE={metrics['f1']:.4f} | "
        f"AP={metrics['average_precision']:.4f} | "
        f"RECALL={metrics['recall']:.4f} | "
        f"PRECISION={metrics['precision']:.4f}"
    )

    print(
        "Modelo salvo:",
        caminho_modelo,
        flush=True
    )


# ============================================================
# 9. INVENTÁRIO FINAL DOS MODELOS
# ============================================================

INVENTARIO = {
    "versao": "2.0",
    "tipo": "condicoes_e_subcategorias",
    "probabilidades_calibradas": True,

    "quantidade_treinados": len(REPORT),
    "quantidade_ignorados": len(IGNORADOS),

    "features": list(FEATURE_COLS),

    "age_groups": list(AGE_GROUPS),

    "sex_encoding": {
        "masculino": 0,
        "feminino": 1
    },

    "modelos": {}
}

for item in REPORT:

    INVENTARIO["modelos"][
        item["modelo"]
    ] = {
        "arquivo": item["arquivo"],
        "categoria": item["categoria"],
        "tipo": item["tipo"],
        "codigo_scp": item["codigo_scp"],
        "threshold": item["threshold"]
    }

(
    DESTINO / "inventario_modelos.json"
).write_text(
    json.dumps(
        INVENTARIO,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 10. RELATÓRIO FINAL
# ============================================================

salvar_relatorios()

report_df = pd.DataFrame(REPORT)
ignored_df = pd.DataFrame(IGNORADOS)

print("\n" + "=" * 65)
print("TREINAMENTO FINALIZADO")
print("=" * 65)

print(
    "Total de modelos candidatos:",
    len(LABELS)
)

print(
    "Modelos treinados:",
    len(REPORT)
)

print(
    "Modelos ignorados:",
    len(IGNORADOS)
)

if not report_df.empty:

    print(
        "F1 médio:",
        round(
            report_df["f1"].mean(),
            4
        )
    )

    print(
        "AP média:",
        round(
            report_df[
                "average_precision"
            ].mean(),
            4
        )
    )

    print(
        "Modelos com F1 = 0:",
        int(
            (report_df["f1"] == 0).sum()
        )
    )

    print(
        "Modelos com F1 >= 0.5:",
        int(
            (report_df["f1"] >= 0.5).sum()
        )
    )

    display(
        report_df.sort_values(
            "f1",
            ascending=False
        ).head(30)
    )

print("\nPasta dos modelos:")
print(DESTINO.resolve())

print("\nArquivos de controle:")
print("metricas_todos_modelos.csv")
print("modelos_nao_treinados.csv")
print("inventario_modelos.json")
print("configuracao_treinamento.json")


AUDITORIA DOS NOMES DE ARQUIVOS
MI__ALMI -> MI_ALMI.joblib
MI__AMI -> MI_AMI.joblib
MI__ASMI -> MI_ASMI.joblib
MI__ILMI -> MI_ILMI.joblib
MI__IMI -> MI_IMI.joblib
MI__INJAL -> MI_INJAL.joblib
MI__INJAS -> MI_INJAS.joblib
MI__INJIL -> MI_INJIL.joblib
MI__INJIN -> MI_INJIN.joblib
MI__INJLA -> MI_INJLA.joblib
MI__IPLMI -> MI_IPLMI.joblib
MI__IPMI -> MI_IPMI.joblib
MI__LMI -> MI_LMI.joblib
MI__PMI -> MI_PMI.joblib
STTC__ANEUR -> STTC_ANEUR.joblib
STTC__DIG -> STTC_DIG.joblib
STTC__EL -> STTC_EL.joblib
STTC__ISCAL -> STTC_ISCAL.joblib
STTC__ISCAN -> STTC_ISCAN.joblib
STTC__ISCAS -> STTC_ISCAS.joblib
STTC__ISCIL -> STTC_ISCIL.joblib
STTC__ISCIN -> STTC_ISCIN.joblib
STTC__ISCLA -> STTC_ISCLA.joblib
STTC__ISC_ -> STTC_ISC_.joblib
STTC__LNGQT -> STTC_LNGQT.joblib
STTC__NDT -> STTC_NDT.joblib
STTC__NST_ -> STTC_NST_.joblib
CD__1AVB -> CD_1AVB.joblib
CD__2AVB -> CD_2AVB.joblib
CD__3AVB -> CD_3AVB.joblib
CD__CLBBB -> CD_CLBBB.joblib
CD__CRBBB -> CD_CRBBB.joblib
CD__ILBBB -> CD_ILBBB.joblib
CD__IRB

,modelo,tipo,categoria,codigo_scp,arquivo,C,threshold,train_pos,valid_pos,test_pos,...,recall,f1,specificity,tn,fp,fn,tp,roc_auc,average_precision,brier_score
40,FORM__PVC,subcategoria,FORM,PVC,FORM_PVC.joblib,3.0,0.302682,908,112,114,...,0.850877,0.779116,0.981308,1995,38,17,97,0.985968,0.768159,0.018954
4,FORM,categoria,FORM,None,FORM.joblib,3.0,0.315040,7121,894,873,...,0.865979,0.649764,0.452119,576,698,117,756,0.745486,0.648211,0.200262
22,CD__CLBBB,subcategoria,CD,CLBBB,CD_CLBBB.joblib,3.0,0.653781,428,54,54,...,0.555556,0.645161,0.995700,2084,9,24,30,0.993630,0.781691,0.010119
1,STTC,categoria,STTC,None,STTC.joblib,3.0,0.224088,4162,523,517,...,0.822050,0.617733,0.733742,1196,434,92,425,0.848015,0.643059,0.126282
0,MI,categoria,MI,None,MI.joblib,3.0,0.300260,4378,540,550,...,0.685455,0.593701,0.785222,1254,343,173,377,0.815942,0.613507,0.142096
2,CD,categoria,CD,None,CD.joblib,3.0,0.276032,3575,459,449,...,0.679287,0.588235,0.833333,1415,283,144,305,0.842193,0.618907,0.116996
29,HYP__LVH,subcategoria,HYP,LVH,HYP_LVH.joblib,1.0,0.387055,1707,210,214,...,0.537383,0.562347,0.958614,1853,80,99,115,0.928606,0.642910,0.055242
7,MI__ASMI,subcategoria,MI,ASMI,MI_ASMI.joblib,3.0,0.179131,1887,236,234,...,0.764957,0.561129,0.882384,1688,225,55,179,0.919498,0.598576,0.063401
3,HYP,categoria,HYP,None,HYP.joblib,1.0,0.231132,2117,267,260,...,0.665385,0.555377,0.899311,1697,190,87,173,0.877217,0.598286,0.071702
26,CD__LAFB,subcategoria,CD,LAFB,CD_LAFB.joblib,3.0,0.230882,1281,163,158,...,0.632911,0.544959,0.945199,1880,109,58,100,0.934641,0.595591,0.044536



Pasta dos modelos:
C:\Users\Matheus\Documents\Modelos\modelos_01

Arquivos de controle:
metricas_todos_modelos.csv
modelos_nao_treinados.csv
inventario_modelos.json
configuracao_treinamento.json



## 9. Inventário de modelos para o visualizador

O inventário identifica:

- Categoria principal
- Subcategoria SCP
- Nome do arquivo `.joblib`
- Características esperadas
- Tipo de probabilidade
- Limiar de classificação

O visualizador deverá carregar esse inventário para
apresentar as subcategorias da condição selecionada.

Os ritmos continuam pertencendo a outro conjunto de
modelos e não são tratados como subcategorias de MI,
STTC, CD, HYP ou FORM.


In [36]:

INVENTARIO = {
    "versao": "1.0",
    "tipo": "condicoes_hierarquicas_multirrotulo",
    "categorias": {},
    "feature_cols": list(FEATURE_COLS),
    "folds": {
        "treino": list(range(1, 9)),
        "calibracao": 9,
        "teste": 10
    },
    "probabilidades": (
        "Calibradas por regressão logística no fold 9"
    ),
    "observacao": (
        "Probabilidades de rótulos SCP; "
        "não representam certeza diagnóstica."
    )
}

trained = {
    item["modelo"]: item
    for item in REPORT
}

for group in GRUPOS:

    general = (
        f"{group}.joblib"
        if group in trained
        else None
    )

    children = []

    for code in sorted(MAPA[group]):
        label = f"{group}__{code}"

        if label not in trained:
            continue

        description = None

        if code in scp.index:
            description = scp.loc[code].get(
                "description"
            )

        children.append({
            "codigo": code,
            "descricao": (
                str(description)
                if pd.notna(description)
                else code
            ),
            "modelo": f"{label}.joblib"
        })

    INVENTARIO["categorias"][group] = {
        "modelo_geral": general,
        "subcategorias": children
    }

with open(
    DESTINO / "inventario_modelos.json",
    "w", encoding="utf-8"
) as f:
    json.dump(
        INVENTARIO, f,
        ensure_ascii=False, indent=2
    )

print("Inventário exportado.")
print(
    "Total de modelos disponíveis:",
    len(REPORT)
)


Inventário exportado.
Total de modelos disponíveis: 44



## 10. Exemplo de inferência contextual

O mesmo ECG pode ser avaliado usando diferentes
idades e sexos.

A aplicação deve:

1. Preservar as características do sinal.
2. Atualizar a idade e o sexo selecionados.
3. Recalcular os indicadores de faixa etária.
4. Reorganizar o vetor na ordem de treinamento.
5. Executar o modelo e o calibrador.
6. Retornar a probabilidade estimada.

Uma nova inferência é obrigatória quando muda o
contexto, mas uma mudança numérica na probabilidade
não é garantida.

As probabilidades de cada subcategoria são independentes.


In [37]:

def predict_artifact(artifact, frame):
    cols = artifact["features"]

    for col in cols:
        if col not in frame.columns:
            raise ValueError(
                f"Característica ausente: {col}"
            )

    x = frame[cols].to_numpy(dtype=float)

    scores = artifact["model"].decision_function(x)

    probabilities = probability_from_scores(
        artifact["calibrator"],
        scores
    )

    return probabilities


def simular_contexto(
    ecg_id,
    idade,
    sexo,
    categoria="MI"
):
    # sexo: 0 masculino, 1 feminino
    if sexo not in (0, 1):
        raise ValueError("Sexo deve ser 0 ou 1.")

    base_row = merged.loc[
        merged.ecg_id == ecg_id
    ].copy()

    if len(base_row) != 1:
        raise ValueError("ECG não encontrado ou duplicado.")

    # Mantém as características reais do ECG.
    base_row["age"] = idade
    base_row["sex_ptbxl"] = sexo

    # Recria indicadores contextuais.
    base_row = contextualize(base_row)

    results = []

    config = INVENTARIO["categorias"][categoria]

    models = []

    if config["modelo_geral"]:
        models.append(config["modelo_geral"])

    models.extend(
        item["modelo"]
        for item in config["subcategorias"]
    )

    for filename in models:
        artifact = joblib.load(
            DESTINO / filename
        )

        p = predict_artifact(
            artifact, base_row
        )[0]

        results.append({
            "modelo": artifact["label"],
            "categoria": artifact["categoria"],
            "codigo_scp": artifact["codigo_scp"],
            "idade": idade,
            "sexo": sexo,
            "probabilidade": float(p),
            "percentual": round(100 * float(p), 2)
        })

    return pd.DataFrame(results)


# Exemplo executável após o treinamento:
#
# ecg_exemplo = int(merged.iloc[0].ecg_id)
#
# display(simular_contexto(
#     ecg_exemplo, idade=25, sexo=0, categoria="MI"
# ))
#
# display(simular_contexto(
#     ecg_exemplo, idade=65, sexo=1, categoria="MI"
# ))



## 11. Interpretação dos resultados e limitações

### O que o notebook entrega

- Um classificador geral por categoria elegível.
- Classificadores individuais por código SCP elegível.
- Probabilidades calibradas de rótulo.
- Características das 12 derivações.
- Características de RR estimadas.
- Idade e sexo.
- Indicadores das quatro faixas etárias.
- Avaliação independente no fold 10.
- Inventário para integração no visualizador.

### O que não deve ser afirmado

- Que cada probabilidade representa certeza clínica.
- Que uma subcategoria foi diagnosticada definitivamente.
- Que todas as subcategorias terão dados suficientes.
- Que mudar idade/sexo sempre mudará a pontuação.
- Que PR, QRS, QTc e ST foram medidos clinicamente
  quando o extrator não fornece essas medidas.

### Integração com o visualizador

Ao selecionar MI, apresentar apenas as probabilidades
dos códigos mapeados para MI.

O mesmo deve ocorrer com STTC, CD, HYP e FORM.

A seleção de uma categoria altera a organização da
exibição, não o sinal nem os rótulos originais.

Os modelos de ritmo devem continuar complementares
e separados.

### Próxima validação necessária

Comparar os modelos com e sem características
demográficas, verificar a calibração por faixa etária
e sexo e avaliar desempenho em pacientes independentes
e, idealmente, em bases externas.
